# Satellite IR CNN — Official Demonstration (Bay of Bengal RI)

This is the **official demo** for the satellite branch of the RI system. It uses the **same canonical code the main pipeline uses** (`src/satellite_cnn.py`):- `RICNNFusion` — hybrid CNN(IR patch) + MLP(tabular) with a valid-pixel mask channel and focal loss.- `predict_image` / `grad_cam` — produce `P(RI 24h)` and a Grad-CAM attention map.- `load_recovered_images` / `normalize_patch` — physical, leakage-safe preprocessing.It imports from `src/` (never duplicates the architecture) and is trained in **Google Colab** (PyTorch/TensorFlow crash on the local macOS box).> **Honesty note.** On a small satellite hold-out, `P(RI)` carries large uncertainty. If the dataset is too small for a reliable model this notebook says so instead of fabricating performance.

## 0. Setup — install torch (in Colab)

In [ ]:
# In Google Colab / a torch-enabled host:# !pip install torch --quietimport sys, ossys.path.insert(0, os.getcwd())  # repo root, so `src` is importable

## 1. Load the canonical model (from src/, not redefined here)

In [ ]:
import numpy as np, pandas as pd, osimport matplotlib.pyplot as pltfrom src.satellite_cnn import (    RICNNFusion, FocalLoss, predict_image, grad_cam,    load_recovered_images, CN_TAB_FEATURES, LAYOUT,)TABULAR_DIM = len(CN_TAB_FEATURES)   # must match the model trained in run_pipeline.pyMODEL_WEIGHTS = "models/satellite_cnn.pt"model = RICNNFusion(tabular_dim=TABULAR_DIM, use_tabular=True)if os.path.exists(MODEL_WEIGHTS):    import torch    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location="cpu"))    print("Loaded canonical weights:", MODEL_WEIGHTS)else:    print("No model weights yet. Train in Colab via run_pipeline.py --satellite,")    print("or with the training cell below, before running the prediction cells.")

## 2. Load the canonical CNN training table (IR + real 11 IMD features)

In [ ]:
# The clean dataset built by src/satellite_cnn.build_cnn_tabular_dataset:# every row has the image AND all 11 contemporaneous IMD features (no imputation).TRAIN_CSV = "results/satellite_cnn_training_data.csv"import osif os.path.exists(TRAIN_CSV):    rows = pd.read_csv(TRAIN_CSV)    print(f"Loaded canonical CNN table: {len(rows)} rows / "          f"{rows['storm_id'].nunique()} storms")    print("11 IMD features:", CN_TAB_FEATURES)    print("Any missing feature:", rows[CN_TAB_FEATURES].isnull().any().any())    print("RI / non-RI:", int((rows["RI_24h"]==1).sum()), "/",          int((rows["RI_24h"]==0).sum()))else:    # Fall back to the raw satellite metadata (only position available).    META = "satellite_cnn_recovered/metadata_clean.csv"    IMAGE_DIR = "satellite_cnn_recovered/images"    rows = pd.read_csv(META)    rows["storm_id"] = rows["storm_id"].astype(str)    if "image_path" not in rows.columns and "image_file" in rows.columns:        rows["image_path"] = rows["image_file"].apply(            lambda f: os.path.join(IMAGE_DIR, f) if not os.path.exists(f) else f)    print("WARN: canonical table missing; using raw metadata (features must be "          "joined before prediction).")rows[["storm_id","datetime_utc","satellite_datetime","RI_24h"]].head()

## 3. Predict P(RI 24h) with the real 11 IMD features

In [ ]:
import numpy as npfrom src.satellite_cnn import _load_fold0_scaler_scale = _load_fold0_scaler("results")   # frozen training scaler (fold 0)def predict_row(row):    p = row["image_path"] if "image_path" in row else None    img = np.load(p) if p and os.path.exists(p) else None    if img is None:        return None, None, None    tb = img[..., 0]    # Build the 11-feature dict (raises if any feature is missing/NaN).    feats = {c: float(row[c]) for c in CN_TAB_FEATURES}    p_ri = predict_image(model, tb, feats, scaler=_scale)    cam = grad_cam(model, tb, feats, scaler=_scale)    return p_ri, cam, imgi = 0row = rows.iloc[i]p_ri, cam, img = predict_row(row)if p_ri is None:    print("Image missing on disk:", row.get("image_path"))else:    thresh = 0.50    label = "RI LIKELY" if p_ri >= thresh else "RI unlikely"    print("="*52)    print(f"Rapid Intensification Probability : {p_ri*100:.1f}%")    print(f"Prediction                        : {label}")    print(f"Threshold                        : {thresh}")    print(f"Storm                            : {row['storm_id']}")    print(f"Valid time                       : {row['datetime_utc']} UTC")    print(f"Satellite time                   : {row.get('satellite_datetime')}")    print(f"Actual RI_24h                    : {int(row['RI_24h'])}")    print("="*52)

## 4. Show the IR patch + Grad-CAM (what the CNN is looking at)

In [ ]:
if p_ri is not None:    fig, axes = plt.subplots(1, 2, figsize=(10, 5))    axes[0].imshow(img[:,:,0], cmap="gray_r", vmin=180, vmax=310)    axes[0].set_title(f"IR storm-centre {row['storm_id']}\nP(RI)={p_ri:.2f} actual={int(row['RI_24h'])}")    axes[0].axis("off")    axes[1].imshow(img[:,:,0], cmap="gray_r", vmin=180, vmax=310)    axes[1].imshow(cam, cmap="jet", alpha=0.55)    axes[1].set_title("Grad-CAM over IR")    axes[1].axis("off")    plt.tight_layout()    plt.savefig("figures/gradcam_demo.png", dpi=130)    plt.show()

## 5. Generate TP / TN / FP / FN Grad-CAM examples

In [ ]:
os.makedirs("figures/gradcam", exist_ok=True)buckets = {"TP": [], "TN": [], "FP": [], "FN": []}for idx in range(len(rows)):    r = rows.iloc[idx]    p = predict_row(r)[0]    if p is None:        continue    y_true = int(r["RI_24h"])    y_pred = 1 if p >= 0.5 else 0    key = str((y_true, y_pred))    label = {"(1, 1)": "TP", "(0, 0)": "TN",             "(1, 0)": "FP", "(0, 1)": "FN"}.get(key)    if label:        buckets[label].append(idx)for key, idxs in buckets.items():    if not idxs:        print(f"  {key}: none on this small set")        continue    i = idxs[0]    p, cam, img = predict_row(rows.iloc[i])    r = rows.iloc[i]    fig, ax = plt.subplots(1, 1, figsize=(4, 4))    ax.imshow(img[:, :, 0], cmap="gray_r", vmin=180, vmax=310)    ax.imshow(cam, cmap="jet", alpha=0.55)    ax.set_title(f"{key} {r['storm_id']}\nP={p:.2f} actual={int(r['RI_24h'])}", fontsize=9)    ax.axis("off")    plt.tight_layout()    plt.savefig(f"figures/gradcam/{key}.png", dpi=120)    plt.show()print("Saved Grad-CAM figures to figures/gradcam/")

## 6. Optional: train the canonical CNN (Colab) and save artifacts

In [ ]:
# Train the canonical storm-safe OOF model and persist the standard artifacts# (satellite_oof_predictions.csv, satellite_embeddings.npy, models/satellite_cnn.pt)# that run_pipeline.py --fusion consumes.## from src.satellite_cnn import run_cnn_oof, extract_embeddings, save_oof_artifacts# import yaml, pandas as pd# cfg = yaml.safe_load(open("config.yaml"))# sat_meta = pd.read_csv(cfg["paths"]["satellite_metadata"])# multimodal = pd.read_csv(cfg["paths"]["multimodal_file"])# res = run_cnn_oof(metadata=sat_meta, multimodal=multimodal, cfg=cfg,#                   seed=cfg["seed"], n_folds=5)# if res["status"] == "trained":#     emb = extract_embeddings(res, cfg)#     wrote = save_oof_artifacts(res, cfg["paths"]["results_dir"], embeddings=emb)#     print("artifacts:", wrote)

## 7. Next steps (SIH)

1. **More storms = more positives.** Prioritise IR for RI-positive fixes (see `tc_ri_cnn/data/download_mergir.py`), then re-train.2. **Multi-frame input** (t, t-6h, t-12h) as 3 channels — already scaffolded by `ir_channels` in `RICNNFusion`; exposes cloud-top cooling trend.3. **Storm-level CV** (grouped-by-storm, mean ± std over folds) — `run_cnn_oof` already does this.4. **Report POD / FAR / CSI**, the metrics RI forecasters use — not just accuracy (meaningless at ~5% positive rate).5. **Fuse embeddings** into the full multimodal system via `run_pipeline.py --fusion` and answer: *does satellite spatial info improve RI beyond IMD+ERA5?*